# KK1 – Palmer Penguins: What separates three penguin species?

**Dataset:** Palmer Penguins — each row is one penguin measured at Palmer Station (Antarctica), 2007–2009.  
**Question:** Which measurements best separate Adelie, Chinstrap, and Gentoo?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Species colors — used in all visualizations below
COLORS = {'Adelie': '#4C72B0', 'Chinstrap': '#DD8452', 'Gentoo': '#55A868'}

print("Libraries loaded.")

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv("penguins.csv")

print("Shape:", df.shape)
print()
df.info()
print()
df.describe()

In [ ]:
# Missing values per column
df.isna().sum()

## 3. Clean the data (with justification)

- **Drop rows** where all measurements are missing (e.g. row 4) — without measurements we cannot analyze.
- **Keep rows** where only `sex` is missing — sex is not needed for the morphology plots.
- **Keep** island and year as they are.

In [ ]:
measure_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

df = df.dropna(subset=measure_cols, how="all").copy()
print(f"Rows after cleaning: {len(df)}")
df.isna().sum()

## 4. Visualizations

### Chart 1 — How many per species?
**Question:** How are the three species distributed?  
**Chart type:** Bar chart (good for counts per category).

In [ ]:
counts = df["species"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values, color=[COLORS[s] for s in counts.index])

ax.set_title("Adelie is the most common species in the dataset")
ax.set_xlabel("Species")
ax.set_ylabel("Number of individuals")
ax.set_ylim(0)

for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 2, str(v), ha="center")

plt.tight_layout()
plt.show()

Adelie has the most observations. All three species are present in the data.

### Chart 2 — Body mass per species
**Question:** Does weight differ between species?  
**Chart type:** Box plot (compare distributions across groups).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

species_list = ["Adelie", "Chinstrap", "Gentoo"]
data = [df.loc[df["species"] == sp, "body_mass_g"].dropna() for sp in species_list]

ax.boxplot(data, labels=species_list, patch_artist=True,
           boxprops=dict(facecolor="lightgray"))

for i, sp in enumerate(species_list, start=1):
    ax.plot([i], [df.loc[df["species"] == sp, "body_mass_g"].mean()],
            marker="D", color=COLORS[sp], markersize=8)

ax.set_title("Gentoo is heaviest — Adelie is lightest")
ax.set_xlabel("Species")
ax.set_ylabel("Body mass (g)")
ax.set_ylim(0)

plt.tight_layout()
plt.show()

df.groupby("species")["body_mass_g"].mean().round(0)

Gentoo is clearly heavier. Adelie and Gentoo are well separated; Chinstrap sits in between.

### Chart 3 — Bill measurements: can we see three groups?
**Question:** Do bill length and bill depth separate the species?  
**Chart type:** Scatter plot (two continuous variables + color by species).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for sp in ["Adelie", "Chinstrap", "Gentoo"]:
    subset = df[df["species"] == sp]
    ax.scatter(
        subset["bill_length_mm"],
        subset["bill_depth_mm"],
        label=sp,
        color=COLORS[sp],
        alpha=0.7,
        s=40,
    )

ax.set_title("Three species form three clear clusters in bill measurements")
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Bill depth (mm)")
ax.legend(title="Species")
ax.set_ylim(0)

plt.tight_layout()
plt.show()

Adelie and Chinstrap overlap slightly, but Gentoo stands apart (longer bill, shallower depth). Bill measurements are useful for telling species apart.

## 5. Conclusion

**What I found:**
- Three species with different body mass (Gentoo heaviest).
- Bill measurements form three groups — Gentoo stands out in particular.
- `groupby` and plots reveal patterns that are hard to see in a table alone.

**What the data cannot tell us:**
- Not all penguins worldwide — only Palmer Station, 2007–2009.
- Missing sex on some rows limits any sex-based analysis.
- Correlation is not causation (these are observations only).